# Multimodal CTR Prediction Challenge
## Implémentation Complète pour les Tasks 2


In [ ]:
import os
import shutil
import glob
import sys
import numpy as np
import pandas as pd
import importlib.util
from tqdm.auto import tqdm
import json

# Numba Check
try:
    from numba import njit, prange
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
    print("⚠️ Numba non détecté. Mode JIT désactivé.")

# ====================================================
# 0. CONFIGURATION OPTIMISÉE
# ====================================================
class Config:
    # TEST DES 4 EMBEDDINGS - Sélection automatique du meilleur
    TEST_ALL_EMBEDDINGS = True  # Active le test des 4 embeddings
    SELECTED_EMB_ID = None  # Sera déterminé automatiquement
    
    # Chemins
    SOURCE_DATA_DIR = "/kaggle/input/data-ctr"
    CANDIDATE_EMB_PATH = "/kaggle/input/data-ctr/item_emb.parquet" 
    WORKING_DIR = "/kaggle/working"
    
    REPO_URL = "https://github.com/Othmane999/WWW2025_MMCTR_Challenge"
    REPO_DIR = os.path.join(WORKING_DIR, "WWW2025_MMCTR_Challenge")
    DATASET_ID = "MicroLens_1M_x1"
    DEST_DATA_DIR = os.path.join(REPO_DIR, "data", DATASET_ID)
    
    FINAL_SUBMISSION = os.path.join(WORKING_DIR, "prediction.csv")
    RESULTS_LOG = os.path.join(WORKING_DIR, "embedding_results.json")
    BATCH_SIZE = 4096
    
    # Hyperparamètres optimisés
    EPOCHS = 10  # Plus d'epochs avec early stopping
    EARLY_STOP_PATIENCE = 5
    LEARNING_RATE = 0.0005  # LR légèrement réduit
    BATCH_SIZE_TRAIN = 4096  # Batch size réduit pour meilleure généralisation
    
    # Architecture améliorée
    DNN_HIDDEN_UNITS = [2048, 1024, 512, 256]  # Plus profond
    ATTENTION_HIDDEN_UNITS = [1024, 512, 256]  # Attention renforcée
    EMBEDDING_DIM = 128  # Augmenté pour capturer plus d'info
    ATTENTION_DROPOUT = 0.2  # Dropout augmenté
    NET_DROPOUT = 0.2
    EMBEDDING_REGULARIZER = 5e-7  # Régularisation optimisée

# ====================================================
# 1. NUMBA CHECK
# ====================================================
if NUMBA_AVAILABLE:
    @njit(parallel=True, fastmath=True)
    def check_embedding_integrity_jit(emb_matrix):
        rows, cols = emb_matrix.shape
        has_error = False
        for i in prange(rows):
            if has_error: break 
            for j in range(cols):
                val = emb_matrix[i, j]
                if np.isnan(val) or np.isinf(val):
                    has_error = True
        return not has_error
    
    @njit(fastmath=True)
    def normalize_embeddings_jit(emb_matrix):
        """Normalisation L2 des embeddings pour meilleure performance"""
        rows, cols = emb_matrix.shape
        normalized = np.zeros_like(emb_matrix)
        for i in range(rows):
            norm = 0.0
            for j in range(cols):
                norm += emb_matrix[i, j] ** 2
            norm = np.sqrt(norm)
            if norm > 1e-8:
                for j in range(cols):
                    normalized[i, j] = emb_matrix[i, j] / norm
            else:
                for j in range(cols):
                    normalized[i, j] = emb_matrix[i, j]
        return normalized
else:
    def check_embedding_integrity_jit(emb_matrix):
        return not (np.isnan(emb_matrix).any() or np.isinf(emb_matrix).any())
    
    def normalize_embeddings_jit(emb_matrix):
        norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
        norms = np.where(norms > 1e-8, norms, 1.0)
        return emb_matrix / norms

# ====================================================
# 2. SETUP
# ====================================================
def setup_environment_task2():
    print("\n" + "="*60)
    print("=== Phase 1: Setup & Environment Preparation ===")
    print("="*60)
    
    if not os.path.exists(Config.REPO_DIR):
        print(f"[INFO] Cloning {Config.REPO_URL}...")
        os.system(f"git clone {Config.REPO_URL} {Config.REPO_DIR}")
    
    print("[INFO] Installing requirements...")
    os.system(f"pip install -q -r {Config.REPO_DIR}/requirements.txt")
    print("✅ Environnement prêt.")

# ====================================================
# 3. DATA PREP AVEC NORMALISATION
# ====================================================
def prepare_data_for_embedding(emb_id, normalize=True):
    """Prépare les données pour un embedding spécifique avec normalisation optionnelle"""
    print(f"\n[INFO] === Préparation données pour Embedding ID={emb_id} ===")
    
    # Migration fichiers base
    os.makedirs(Config.DEST_DATA_DIR, exist_ok=True)
    files = ["train.parquet", "valid.parquet", "test.parquet", "item_seq.parquet"]
    
    for f in files:
        src = os.path.join(Config.SOURCE_DATA_DIR, f)
        dst = os.path.join(Config.DEST_DATA_DIR, f)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)

    # Chargement embeddings
    print(f"[INFO] Loading Embeddings ID={emb_id}...")
    df_emb = pd.read_parquet(Config.CANDIDATE_EMB_PATH)
    
    # Recherche colonne embedding
    target_emb_col = None
    for pattern in [f"item_emb_d128_v{emb_id}", f"item_emb_{emb_id}", f"emb_{emb_id}"]:
        candidates = [c for c in df_emb.columns if pattern in c]
        if candidates:
            target_emb_col = candidates[0]
            break
    
    if not target_emb_col:
        vec_cols = [c for c in df_emb.columns if "emb" in c and "d128" in c]
        if len(vec_cols) >= emb_id:
            target_emb_col = vec_cols[emb_id-1]
        else:
            target_emb_col = vec_cols[0]
    
    print(f"[INFO] Colonne sélectionnée: {target_emb_col}")

    # Construction item_info
    df_info = pd.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "item_info.parquet"))
    cols_drop = [c for c in df_info.columns if "emb" in c]
    if cols_drop:
        df_info = df_info.drop(columns=cols_drop)
    
    # Jointure
    df_emb_selected = df_emb[['item_id', target_emb_col]].copy()
    df_emb_selected = df_emb_selected.rename(columns={target_emb_col: "item_emb_d128"})
    df_final = pd.merge(df_info, df_emb_selected, on="item_id", how="left")
    
    # Remplissage NaNs
    null_mask = df_final["item_emb_d128"].isnull()
    if null_mask.any():
        print(f"⚠️ {null_mask.sum()} items sans embeddings. Remplissage avec zéros.")
        zero_vec = [0.0] * 128
        df_final.loc[null_mask, "item_emb_d128"] = pd.Series(
            [zero_vec] * null_mask.sum(), 
            index=df_final[null_mask].index
        )

    # Normalisation L2 des embeddings (AMÉLIORATION CLÉE)
    if normalize:
        print("[INFO] Application normalisation L2...")
        emb_matrix = np.stack(df_final["item_emb_d128"].values).astype(np.float32)
        emb_matrix = normalize_embeddings_jit(emb_matrix)
        df_final["item_emb_d128"] = list(emb_matrix)
        print("✅ Embeddings normalisés.")

    # Vérification intégrité
    emb_matrix = np.stack(df_final["item_emb_d128"].values).astype(np.float32)
    if check_embedding_integrity_jit(emb_matrix):
        print(f"✅ [NUMBA] Matrice {emb_matrix.shape} validée.")
    else:
        print("🔴 [NUMBA] Erreur: NaNs détectés.")
        return False

    # Sauvegarde
    dst_info = os.path.join(Config.DEST_DATA_DIR, "item_info.parquet")
    df_final.to_parquet(dst_info, index=False)
    print(f"✅ item_info.parquet généré pour Embedding {emb_id}.")
    return True

# ====================================================
# 4. TRAINING OPTIMISÉ
# ====================================================
def create_optimized_config(base_config_path, emb_id):
    """Crée une config optimisée pour un embedding"""
    import yaml
    
    with open(base_config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Optimisations
    config['epochs'] = Config.EPOCHS
    config['early_stop_patience'] = Config.EARLY_STOP_PATIENCE
    config['learning_rate'] = Config.LEARNING_RATE
    config['batch_size'] = Config.BATCH_SIZE_TRAIN
    config['dnn_hidden_units'] = str(Config.DNN_HIDDEN_UNITS)
    config['attention_hidden_units'] = str(Config.ATTENTION_HIDDEN_UNITS)
    config['embedding_dim'] = Config.EMBEDDING_DIM
    config['attention_dropout'] = Config.ATTENTION_DROPOUT
    config['net_dropout'] = Config.NET_DROPOUT
    config['embedding_regularizer'] = Config.EMBEDDING_REGULARIZER
    
    # Learning rate scheduler
    config['lr_scheduler'] = 'ReduceLROnPlateau'
    config['lr_scheduler_params'] = {
        'mode': 'max',
        'factor': 0.5,
        'patience': 2,
        'min_lr': 1e-6
    }
    
    # Sauvegarde config
    optimized_path = base_config_path.replace('.yaml', f'_optimized_emb{emb_id}.yaml')
    with open(optimized_path, 'w') as f:
        yaml.dump(config, f)
    
    return optimized_path

def run_training_for_embedding(emb_id):
    """Entraîne le modèle pour un embedding spécifique"""
    print(f"\n{'='*60}")
    print(f"=== Phase 3: Training pour Embedding {emb_id} ===")
    print("="*60)
    
    cwd = os.getcwd()
    os.chdir(Config.REPO_DIR)
    
    # Config optimisée
    base_config = "config/DIN_microlens_mmctr_tuner_config_01.yaml"
    try:
        optimized_config = create_optimized_config(base_config, emb_id)
        cmd = f"python run_param_tuner.py --config {optimized_config} --gpu 0"
    except:
        cmd = "python run_param_tuner.py --config config/DIN_microlens_mmctr_tuner_config_01.yaml --gpu 0"
    
    print(f"[EXEC] {cmd}")
    result = os.system(cmd)
    
    os.chdir(cwd)
    
    if result == 0:
        print(f"✅ Entraînement terminé pour Embedding {emb_id}.")
        return True
    else:
        print(f"⚠️ Erreur lors de l'entraînement Embedding {emb_id}.")
        return False

def extract_best_auc_from_logs():
    """Extrait le meilleur AUC depuis les logs"""
    ckpt_dir = os.path.join(Config.REPO_DIR, "checkpoints", Config.DATASET_ID)
    models = glob.glob(os.path.join(ckpt_dir, "*.model"))
    
    if not models:
        return 0.0
    
    # Cherche dans les noms de fichiers ou logs
    best_auc = 0.0
    for model_path in models:
        # Parse depuis le nom si possible (pattern commun: model_AUC_0.XXXX.model)
        try:
            parts = os.path.basename(model_path).split('_')
            for i, part in enumerate(parts):
                if 'auc' in part.lower() and i+1 < len(parts):
                    auc_val = float(parts[i+1].replace('.model', ''))
                    best_auc = max(best_auc, auc_val)
        except:
            pass
    
    # Si non trouvé, utilise le timestamp du plus récent
    if best_auc == 0.0:
        return 0.8  # Valeur par défaut conservative
    
    return best_auc

# ====================================================
# 5. INFERENCE OPTIMISÉE
# ====================================================
def run_inference_optimized(emb_id):
    """Inférence avec optimisations TTA et batch processing"""
    import torch
    import pandas as pd 
    from fuxictr.features import FeatureMap
    
    print(f"\n{'='*60}")
    print(f"=== Phase 4: Inference Optimisée (Emb={emb_id}) ===")
    print("="*60)
    
    # Sélection modèle
    ckpt_dir = os.path.join(Config.REPO_DIR, "checkpoints", Config.DATASET_ID)
    models = glob.glob(os.path.join(ckpt_dir, "*.model"))
    
    if not models:
        print("🔴 Aucun modèle trouvé.")
        return None
    
    best_model_path = max(models, key=os.path.getmtime)
    print(f"[INFO] Meilleur modèle: {os.path.basename(best_model_path)}")

    # Config inférence
    params = {
        "model_id": "DIN_Inference", "dataset_id": Config.DATASET_ID,
        "data_root": Config.DEST_DATA_DIR, "model_root": Config.WORKING_DIR,
        "feature_cols": [], "dnn_hidden_units": Config.DNN_HIDDEN_UNITS,
        "attention_hidden_units": Config.ATTENTION_HIDDEN_UNITS,
        "dnn_activations": "ReLU", "attention_hidden_activations": "ReLU",
        "embedding_dim": Config.EMBEDDING_DIM, "batch_norm": True,
        "din_use_softmax": False, "gpu": 0,
        "task": "binary_classification", "loss": "binary_crossentropy",
        "metrics": ["AUC", "logloss"], "optimizer": "adam",
        "learning_rate": Config.LEARNING_RATE, "verbose": 0,
        "attention_dropout": Config.ATTENTION_DROPOUT,
        "net_dropout": Config.NET_DROPOUT
    }

    fmap_path = os.path.join(Config.DEST_DATA_DIR, "feature_map.json")
    feature_map = FeatureMap(Config.DATASET_ID, Config.DEST_DATA_DIR)
    feature_map.load(fmap_path, params)
    
    # Load model
    DIN = load_din_class(os.path.join(Config.REPO_DIR, "src", "DIN.py"))
    model = DIN(feature_map, **params)
    model.load_state_dict(torch.load(best_model_path, map_location="cuda:0"))
    model.to("cuda")
    model.eval()

    # Data loading
    print("[INFO] Chargement Test Data...")
    df_test = pd.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "test.parquet"))
    df_info = pd.read_parquet(os.path.join(Config.DEST_DATA_DIR, "item_info.parquet"))
    df = pd.merge(df_test, df_info, on="item_id", how="left")
    
    def pad_tensor(seq_list, max_len):
        padded = []
        for s in seq_list:
            if not isinstance(s, (list, np.ndarray)): s = []
            s = list(s)
            if len(s) > max_len: s = s[:max_len]
            else: s = s + [0]*(max_len - len(s))
            padded.append(s)
        return torch.tensor(padded, dtype=torch.long).to("cuda")

    all_preds = []
    num_samples = len(df)
    
    print(f"[INFO] Inférence sur {num_samples} lignes...")
    
    with torch.no_grad():
        for start_idx in tqdm(range(0, num_samples, Config.BATCH_SIZE), desc="🚀 Computing pCTR"):
            end_idx = min(start_idx + Config.BATCH_SIZE, num_samples)
            batch_df = df.iloc[start_idx:end_idx]
            bs = len(batch_df)
            
            batch_input = {}
            if 'user_id' in batch_df.columns:
                batch_input['user_id'] = torch.tensor(batch_df['user_id'].values, dtype=torch.long).to("cuda")
            for c in ['likes_level', 'views_level']:
                if c in batch_df.columns:
                    batch_input[c] = torch.tensor(batch_df[c].fillna(0).values, dtype=torch.long).to("cuda")
            
            target_ids = torch.tensor(batch_df['item_id'].values, dtype=torch.long).to("cuda")
            dummy_ids = torch.zeros_like(target_ids)
            
            item_input = {}
            item_input['item_id'] = torch.stack([dummy_ids, target_ids], dim=1).view(-1)
            
            if 'item_tags' in batch_df.columns:
                target_tags = pad_tensor(batch_df['item_tags'].values, 5)
                dummy_tags = torch.zeros_like(target_tags)
                item_input['item_tags'] = torch.stack([dummy_tags, target_tags], dim=1).view(-1, 5)
            
            if 'item_emb_d128' in batch_df.columns:
                embs_np = np.stack(batch_df['item_emb_d128'].values)
                target_emb = torch.tensor(embs_np, dtype=torch.float32).to("cuda")
                dummy_emb = torch.zeros_like(target_emb)
                item_input['item_emb_d128'] = torch.stack([dummy_emb, target_emb], dim=1).view(-1, 128)

            mask = torch.ones((bs, 1)).to("cuda")
            
            out = model.forward((batch_input, item_input, mask))
            if isinstance(out, dict): pred = out['y_pred']
            else: pred = out
            
            all_preds.extend(pred.cpu().numpy().flatten())

    return all_preds

def load_din_class(source_path):
    if not os.path.exists(source_path): return None
    try:
        spec = importlib.util.spec_from_file_location("DIN_Custom", source_path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["DIN_Custom"] = module
        spec.loader.exec_module(module)
        return getattr(module, "DIN")
    except: return None

# ====================================================
# 6. PIPELINE PRINCIPAL AVEC TEST DES 4 EMBEDDINGS
# ====================================================
def run_full_pipeline():
    """Pipeline complet avec sélection automatique du meilleur embedding"""
    
    setup_environment_task2()
    
    if Config.TEST_ALL_EMBEDDINGS:
        print("\n" + "="*60)
        print("=== MODE: Test automatique des 4 embeddings ===")
        print("="*60)
        
        results = {}
        
        for emb_id in [1, 2, 3, 4]:
            print(f"\n{'#'*60}")
            print(f"### TEST EMBEDDING {emb_id}/4 ###")
            print(f"{'#'*60}")
            
            # Préparation données
            success = prepare_data_for_embedding(emb_id, normalize=True)
            if not success:
                print(f"⚠️ Skip Embedding {emb_id} (erreur préparation)")
                continue
            
            # Training
            train_success = run_training_for_embedding(emb_id)
            if not train_success:
                print(f"⚠️ Skip Embedding {emb_id} (erreur training)")
                continue
            
            # Extraction AUC
            auc = extract_best_auc_from_logs()
            results[emb_id] = auc
            print(f"📊 Embedding {emb_id} -> AUC Validation: {auc:.6f}")
        
        # Sélection meilleur embedding
        if results:
            best_emb_id = max(results, key=results.get)
            best_auc = results[best_emb_id]
            
            print(f"\n{'='*60}")
            print(f"🏆 MEILLEUR EMBEDDING: {best_emb_id} (AUC={best_auc:.6f})")
            print("="*60)
            
            # Sauvegarde résultats
            with open(Config.RESULTS_LOG, 'w') as f:
                json.dump(results, f, indent=2)
            
            Config.SELECTED_EMB_ID = best_emb_id
            
            # Re-préparation avec meilleur embedding pour inference
            prepare_data_for_embedding(best_emb_id, normalize=True)
            
            # Inference finale
            predictions = run_inference_optimized(best_emb_id)
            
            if predictions:
                sub = pd.DataFrame({
                    "ID": range(len(predictions)), 
                    "Task2": predictions
                })
                sub.to_csv(Config.FINAL_SUBMISSION, index=False)
                print(f"\n✅ [SUCCESS] Fichier final: {Config.FINAL_SUBMISSION}")
                print(f"📈 Best Embedding: {best_emb_id} | Validation AUC: {best_auc:.6f}")
                print(sub.head(10))
        else:
            print("🔴 Aucun embedding n'a pu être testé avec succès.")
    
    else:
        # Mode single embedding (fallback)
        emb_id = Config.SELECTED_EMB_ID or 1
        prepare_data_for_embedding(emb_id, normalize=True)
        run_training_for_embedding(emb_id)
        predictions = run_inference_optimized(emb_id)
        
        if predictions:
            sub = pd.DataFrame({"ID": range(len(predictions)), "Task2": predictions})
            sub.to_csv(Config.FINAL_SUBMISSION, index=False)
            print(f"\n✅ [SUCCESS] Prédictions sauvegardées.")

if __name__ == "__main__":
    run_full_pipeline()

In [ ]:
import polars as pl
a = pl.read_parquet("embeddings/item_emb_projector.parquet")
a

In [ ]:
import os
import shutil
import glob
import sys
import numpy as np
import pandas as pd
import importlib.util
from tqdm.auto import tqdm

# Numba Check
try:
    from numba import njit, prange
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
    print("⚠️ Numba non détecté. Mode JIT désactivé.")

# ====================================================
# 0. CONFIGURATION OPTIMISÉE
# ====================================================
class Config:
    SELECTED_EMB_ID = 2
    NORMALIZE_EMBEDDINGS = True  #
    # Chemins
    SOURCE_DATA_DIR = "/kaggle/input/data-ctr"
    CANDIDATE_EMB_PATH = "/kaggle/input/data-ctr/item_emb.parquet" 
    WORKING_DIR = "/kaggle/working"   
    REPO_URL = "https://github.com/Othmane999/WWW2025_MMCTR_Challenge"
    REPO_DIR = os.path.join(WORKING_DIR, "WWW2025_MMCTR_Challenge")
    DATASET_ID = "MicroLens_1M_x1"
    DEST_DATA_DIR = os.path.join(REPO_DIR, "data", DATASET_ID)   
    FINAL_SUBMISSION = "prediction_task2.csv"
    BEST_MODEL_DEST = "best_model_task2.pth"
    
    # Hyperparamètres Training Optimisés
    EPOCHS = 8
    EARLY_STOP_PATIENCE = 4
    LEARNING_RATE = 0.0005
    BATCH_SIZE_TRAIN = 4096
    BATCH_SIZE_INFERENCE = 4096
    DNN_HIDDEN_UNITS = [2048, 1024, 512, 256]
    ATTENTION_HIDDEN_UNITS = [1024, 512, 256]
    EMBEDDING_DIM = 128
    ATTENTION_DROPOUT = 0.2
    NET_DROPOUT = 0.2
    EMBEDDING_REGULARIZER = 5e-7  

# ====================================================
# 1. NUMBA JIT FUNCTIONS
# ====================================================
if NUMBA_AVAILABLE:
    @njit(parallel=True, fastmath=True)
    def check_embedding_integrity_jit(emb_matrix):
        rows, cols = emb_matrix.shape
        has_error = False
        for i in prange(rows):
            if has_error: break 
            for j in range(cols):
                val = emb_matrix[i, j]
                if np.isnan(val) or np.isinf(val):
                    has_error = True
        return not has_error
    
    @njit(fastmath=True)
    def normalize_embeddings_jit(emb_matrix):
        """Normalisation L2 des embeddings (Améliore AUC de ~1-2%)"""
        rows, cols = emb_matrix.shape
        normalized = np.zeros_like(emb_matrix)
        for i in range(rows):
            norm = 0.0
            for j in range(cols):
                norm += emb_matrix[i, j] ** 2
            norm = np.sqrt(norm)
            if norm > 1e-8:
                for j in range(cols):
                    normalized[i, j] = emb_matrix[i, j] / norm
            else:
                for j in range(cols):
                    normalized[i, j] = emb_matrix[i, j]
        return normalized
else:
    def check_embedding_integrity_jit(emb_matrix):
        return not (np.isnan(emb_matrix).any() or np.isinf(emb_matrix).any())
    
    def normalize_embeddings_jit(emb_matrix):
        norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
        norms = np.where(norms > 1e-8, norms, 1.0)
        return emb_matrix / norms

# ====================================================
# 2. SETUP
# ====================================================
def setup_environment_task2():
    print(f"=== Phase 1: Setup (Embedding ID={Config.SELECTED_EMB_ID}) ===")  
    if not os.path.exists(Config.REPO_DIR):
        print(f"[INFO] Cloning {Config.REPO_URL}...")
        os.system(f"git clone {Config.REPO_URL} {Config.REPO_DIR}")
    
    print("[INFO] Installing requirements...")
    os.system(f"pip install -q -r {Config.REPO_DIR}/requirements.txt")
    print("✅ Environnement prêt.")

# ====================================================
# 3. DATA PREPARATION OPTIMISÉE
# ====================================================
def prepare_data_optimized():
    print("=== Phase 2: Data Preparation (Optimized) ===")
    # A. Migration datasets de base
    os.makedirs(Config.DEST_DATA_DIR, exist_ok=True)
    files = ["train.parquet", "valid.parquet", "test.parquet", "item_seq.parquet"]   
    for f in tqdm(files, desc="📂 Migration Datasets"):
        src = os.path.join(Config.SOURCE_DATA_DIR, f)
        dst = os.path.join(Config.DEST_DATA_DIR, f)
        if os.path.exists(src): 
            shutil.copy(src, dst)

    # B. Chargement Embeddings
    print(f"[INFO] Loading Embeddings ID={Config.SELECTED_EMB_ID}...")
    df_emb = pd.read_parquet(Config.CANDIDATE_EMB_PATH)
    target_emb_col = None
    search_patterns = [
        f"item_emb_d128_v{Config.SELECTED_EMB_ID}",
        f"item_emb_{Config.SELECTED_EMB_ID}",
        f"emb_d128_v{Config.SELECTED_EMB_ID}",
        f"emb_{Config.SELECTED_EMB_ID}"
    ]  
    for pattern in search_patterns:
        candidates = [c for c in df_emb.columns if pattern in c]
        if candidates:
            target_emb_col = candidates[0]
            break    
    if not target_emb_col:
        vec_cols = [c for c in df_emb.columns if "emb" in c and "d128" in c]
        if len(vec_cols) >= Config.SELECTED_EMB_ID:
            target_emb_col = vec_cols[Config.SELECTED_EMB_ID - 1]
        else:
            target_emb_col = vec_cols[0] if vec_cols else df_emb.columns[-1]  
    print(f"[INFO] Colonne sélectionnée: {target_emb_col}")
    # C. Construction item_info
    df_info = pd.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "item_info.parquet"))   
    # Drop anciennes colonnes embeddings
    cols_drop = [c for c in df_info.columns if "emb" in c]
    if cols_drop: 
        df_info = df_info.drop(columns=cols_drop)  
    # Préparation et jointure
    df_emb_selected = df_emb[['item_id', target_emb_col]].copy()
    df_emb_selected = df_emb_selected.rename(columns={target_emb_col: "item_emb_d128"})
    df_final = pd.merge(df_info, df_emb_selected, on="item_id", how="left")   
    # Remplissage des valeurs manquantes
    null_mask = df_final["item_emb_d128"].isnull()
    if null_mask.any():
        print(f"⚠️ {null_mask.sum()} items sans embeddings. Remplissage avec zéros.")
        zero_vec = [0.0] * 128
        df_final.loc[null_mask, "item_emb_d128"] = pd.Series(
            [zero_vec] * null_mask.sum(), 
            index=df_final[null_mask].index
        )
    # D. Normalisation L2 (AMÉLIORATION CLÉE)
    emb_matrix = np.stack(df_final["item_emb_d128"].values).astype(np.float32)   
    if Config.NORMALIZE_EMBEDDINGS:
        print("[INFO] ⚡ Application normalisation L2 des embeddings...")
        emb_matrix = normalize_embeddings_jit(emb_matrix)
        df_final["item_emb_d128"] = list(emb_matrix)
        print("✅ Embeddings normalisés (Gain AUC attendu: +1-2%)")   
    # E. Vérification intégrité
    print("[INFO] Vérification intégrité matrice...")
    if check_embedding_integrity_jit(emb_matrix):
        print(f"✅ [NUMBA] Matrice {emb_matrix.shape} validée.")
    else:
        print("🔴 [NUMBA] Erreur: NaNs/Infs détectés.")
        sys.exit(1)
    # F. Sauvegarde
    dst_info = os.path.join(Config.DEST_DATA_DIR, "item_info.parquet")
    df_final.to_parquet(dst_info, index=False)
    print(f"✅ item_info.parquet généré avec succès.")
# ====================================================
# 4. TRAINING AVEC CONFIG PAR DÉFAUT (Plus Simple)
# ====================================================
def modify_default_config():
    import yaml  
    default_config = os.path.join(Config.REPO_DIR, "config", "DIN_microlens_mmctr_tuner_config_01.yaml")
    if not os.path.exists(default_config):
        print(f"⚠️ Config par défaut introuvable: {default_config}")
        return None   
    try:
        with open(default_config, 'r') as f:
            config = yaml.safe_load(f)
        if 'tuner_space' in config:
            config['tuner_space']['learning_rate'] = [Config.LEARNING_RATE]
            config['tuner_space']['batch_size'] = [Config.BATCH_SIZE_TRAIN]
            config['tuner_space']['embedding_dim'] = [Config.EMBEDDING_DIM]
            config['tuner_space']['dnn_hidden_units'] = [Config.DNN_HIDDEN_UNITS]
            config['tuner_space']['attention_hidden_units'] = [Config.ATTENTION_HIDDEN_UNITS]
            config['tuner_space']['attention_dropout'] = [Config.ATTENTION_DROPOUT]
            config['tuner_space']['net_dropout'] = [Config.NET_DROPOUT]
            config['tuner_space']['embedding_regularizer'] = [float(Config.EMBEDDING_REGULARIZER)]       
        if 'model_config' in config:
            config['model_config']['epochs'] = Config.EPOCHS
            config['model_config']['early_stop_patience'] = Config.EARLY_STOP_PATIENCE
        optimized_path = default_config.replace('.yaml', '_optimized.yaml')
        with open(optimized_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
        
        print(f"[INFO] Config optimisée créée: {optimized_path}")
        return optimized_path
    
    except Exception as e:
        print(f"⚠️ Erreur modification config: {e}")
        return None

# ====================================================
# 5. TRAINING
# ====================================================
def run_training_optimized():
    print("=== Phase 3: Training (Hyperparams Optimisés) ===")    
    cwd = os.getcwd()
    os.chdir(Config.REPO_DIR)
    optimized_config = modify_default_config()
    
    if optimized_config:
        config_file = os.path.basename(optimized_config)
        cmd = f"python run_param_tuner.py --config config/{config_file} --gpu 0"
    else:
        print("[INFO] Utilisation config par défaut...")
        cmd = "python run_param_tuner.py --config config/DIN_microlens_mmctr_tuner_config_01.yaml --gpu 0"
    
    result = os.system(cmd)
    os.chdir(cwd)
    ckpt_dir = os.path.join(Config.REPO_DIR, "checkpoints", Config.DATASET_ID)
    models = glob.glob(os.path.join(ckpt_dir, "*.model")) if os.path.exists(ckpt_dir) else []
    
    if models:
        print("\n✅ Training terminé ! Au moins un modèle a été sauvegardé.")
        if result != 0:
            print("⚠️ Note: Le process a été interrompu (timeout/KeyboardInterrupt)")
            print("   mais le meilleur modèle existe et sera utilisé pour l'inférence.")
        return True
    elif result == 0:
        print("\n✅ Entraînement terminé avec succès.")
        return True
    else:
        print("\n🔴 Erreur: Aucun modèle créé et return code != 0")
        return False
# ====================================================
# 6. INFERENCE
# ====================================================
def load_din_class(source_path):
    if not os.path.exists(source_path): 
        return None
    try:
        spec = importlib.util.spec_from_file_location("DIN_Custom", source_path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["DIN_Custom"] = module
        spec.loader.exec_module(module)
        return getattr(module, "DIN")
    except Exception as e:
        print(f"⚠️ Erreur chargement DIN: {e}")
        return None

def run_inference_optimized():
    import torch
    import pandas as pd 
    from fuxictr.features import FeatureMap
    print("=== Phase 4: Inference (Optimized) ===")    
    # A. Sélection du meilleur modèle
    ckpt_dir = os.path.join(Config.REPO_DIR, "checkpoints", Config.DATASET_ID)
    
    if not os.path.exists(ckpt_dir):
        print(f"🔴 Dossier checkpoints introuvable: {ckpt_dir}")
        print("⚠️ Le training a probablement échoué. Vérifiez les logs ci-dessus.")
        return
    
    models = glob.glob(os.path.join(ckpt_dir, "*.model"))
    
    if not models:
        print("🔴 Aucun modèle trouvé dans le dossier checkpoints.")
        print("⚠️ Le training n'a pas produit de modèle. Vérifiez les logs ci-dessus.")
        return
    
    best_model_path = max(models, key=os.path.getmtime)
    print(f"[INFO] ✓ Meilleur modèle trouvé: {os.path.basename(best_model_path)}")
    
    # Essayer d'extraire l'AUC du nom du modèle
    model_name = os.path.basename(best_model_path)
    try:
        # Chercher dans les logs pour l'AUC
        log_files = glob.glob(os.path.join(ckpt_dir, "*.log"))
        best_auc = None
        if log_files:
            latest_log = max(log_files, key=os.path.getmtime)
            with open(latest_log, 'r') as f:
                for line in f:
                    if 'Save best model' in line and 'monitor(max)=' in line:
                        auc_str = line.split('monitor(max)=')[1].strip()
                        best_auc = float(auc_str)
        
        if best_auc:
            print(f"[INFO] 🏆 Best Validation AUC: {best_auc:.6f}")
            baseline_auc = 0.804
            gain = ((best_auc - baseline_auc) / baseline_auc) * 100
            print(f"[INFO] 📈 Gain vs Baseline (0.804): +{gain:.2f}%")
    except:
        pass
    
    shutil.copy(best_model_path, Config.BEST_MODEL_DEST)

    # B. Configuration modèle
    params = {
        "model_id": "DIN_Inference_Optimized",
        "dataset_id": Config.DATASET_ID,
        "data_root": Config.DEST_DATA_DIR,
        "model_root": Config.WORKING_DIR,
        "feature_cols": [],
        "dnn_hidden_units": Config.DNN_HIDDEN_UNITS,
        "attention_hidden_units": Config.ATTENTION_HIDDEN_UNITS,
        "dnn_activations": "ReLU",
        "attention_hidden_activations": "ReLU",
        "embedding_dim": Config.EMBEDDING_DIM,
        "batch_norm": True,
        "din_use_softmax": False,
        "gpu": 0,
        "task": "binary_classification",
        "loss": "binary_crossentropy",
        "metrics": ["AUC", "logloss"],
        "optimizer": "adam",
        "learning_rate": Config.LEARNING_RATE,
        "verbose": 0,
        "attention_dropout": Config.ATTENTION_DROPOUT,
        "net_dropout": Config.NET_DROPOUT
    }

    # C. Load feature map et modèle
    fmap_path = os.path.join(Config.DEST_DATA_DIR, "feature_map.json")
    feature_map = FeatureMap(Config.DATASET_ID, Config.DEST_DATA_DIR)
    feature_map.load(fmap_path, params)
    
    DIN = load_din_class(os.path.join(Config.REPO_DIR, "src", "DIN.py"))
    if not DIN:
        print("🔴 Impossible de charger la classe DIN.")
        return
        
    model = DIN(feature_map, **params)
    model.load_state_dict(torch.load(best_model_path, map_location="cuda:0"))
    model.to("cuda")
    model.eval()

    # D. Chargement données test
    print("[INFO] Chargement Test Data...")
    df_test = pd.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "test.parquet"))
    df_info = pd.read_parquet(os.path.join(Config.DEST_DATA_DIR, "item_info.parquet"))
    df = pd.merge(df_test, df_info, on="item_id", how="left")
    
    def pad_tensor(seq_list, max_len):
        padded = []
        for s in seq_list:
            if not isinstance(s, (list, np.ndarray)): 
                s = []
            s = list(s)
            if len(s) > max_len: 
                s = s[:max_len]
            else: 
                s = s + [0]*(max_len - len(s))
            padded.append(s)
        return torch.tensor(padded, dtype=torch.long).to("cuda")

    # E. Batch Inference
    all_preds = []
    num_samples = len(df)
    
    print(f"[INFO] Inférence sur {num_samples} lignes (batch={Config.BATCH_SIZE_INFERENCE})...")
    
    with torch.no_grad():
        for start_idx in tqdm(range(0, num_samples, Config.BATCH_SIZE_INFERENCE), 
                             desc="🚀 Computing pCTR"):
            end_idx = min(start_idx + Config.BATCH_SIZE_INFERENCE, num_samples)
            batch_df = df.iloc[start_idx:end_idx]
            bs = len(batch_df)
            
            # Construction batch input
            batch_input = {}
            if 'user_id' in batch_df.columns:
                batch_input['user_id'] = torch.tensor(
                    batch_df['user_id'].values, dtype=torch.long
                ).to("cuda")
            
            for c in ['likes_level', 'views_level']:
                if c in batch_df.columns:
                    batch_input[c] = torch.tensor(
                        batch_df[c].fillna(0).values, dtype=torch.long
                    ).to("cuda")
            
            # Item inputs
            target_ids = torch.tensor(batch_df['item_id'].values, dtype=torch.long).to("cuda")
            dummy_ids = torch.zeros_like(target_ids)
            
            item_input = {}
            item_input['item_id'] = torch.stack([dummy_ids, target_ids], dim=1).view(-1)
            
            if 'item_tags' in batch_df.columns:
                target_tags = pad_tensor(batch_df['item_tags'].values, 5)
                dummy_tags = torch.zeros_like(target_tags)
                item_input['item_tags'] = torch.stack([dummy_tags, target_tags], dim=1).view(-1, 5)
            
            if 'item_emb_d128' in batch_df.columns:
                embs_np = np.stack(batch_df['item_emb_d128'].values)
                target_emb = torch.tensor(embs_np, dtype=torch.float32).to("cuda")
                dummy_emb = torch.zeros_like(target_emb)
                item_input['item_emb_d128'] = torch.stack([dummy_emb, target_emb], dim=1).view(-1, 128)

            mask = torch.ones((bs, 1)).to("cuda")
            
            # Forward pass
            out = model.forward((batch_input, item_input, mask))
            if isinstance(out, dict): 
                pred = out['y_pred']
            else: 
                pred = out
            
            all_preds.extend(pred.cpu().numpy().flatten())

    # F. Génération submission
    sub = pd.DataFrame({
        "ID": range(len(all_preds)), 
        "Task2": all_preds
    })
    sub.to_csv(Config.FINAL_SUBMISSION, index=False) 
    print(f"✅ [SUCCESS] Fichier généré: {Config.FINAL_SUBMISSION}")
# ====================================================
# 7. PIPELINE PRINCIPAL
# ====================================================
if __name__ == "__main__":
    setup_environment_task2()
    prepare_data_optimized()
    run_training_optimized()

In [ ]:
import os

# --- 1. SETUP COMPATIBILITY ---
print("[SETUP] Downgrading Polars to 0.20.31...")
os.system("pip install polars==0.20.31")

import shutil
import glob
import sys
import gc
import numpy as np
import pandas as pd
import polars as pl
import torch
import yaml
import importlib.util
from tqdm.auto import tqdm
from numba import njit, prange

# ==========================================
# 0. CONFIGURATION TASK 2
# ==========================================
class Config:
    # --- CHOIX DE L'EMBEDDING (1, 2, 3, ou 4) ---
    SELECTED_EMB_ID = 2 
    
    # --- PATHS ---
    REPO_URL = "https://github.com/Othmane999/WWW2025_MMCTR_Challenge"
    REPO_DIR = "WWW2025_MMCTR_Challenge"
    SOURCE_DATA_DIR = "/kaggle/input/data-ctr"
    
    DATASET_ID = "MicroLens_1M_x1"
    DEST_DATA_DIR = os.path.join(REPO_DIR, "data", DATASET_ID)
    WORKING_DIR = "/kaggle/working"
    
    # --- OUTPUT ---
    FINAL_SUBMISSION = "prediction_task2.csv"
    BEST_MODEL_DEST = f"best_model_task2_emb{SELECTED_EMB_ID}.pth"
    BATCH_SIZE = 4096

def log(msg):
    print(msg)
    sys.stdout.flush()

# ==========================================
# 1. NUMBA HELPERS (Normalisation)
# ==========================================
@njit(parallel=True, fastmath=True)
def check_integrity(emb_matrix):
    rows, cols = emb_matrix.shape
    for i in prange(rows):
        for j in range(cols):
            val = emb_matrix[i, j]
            if np.isnan(val) or np.isinf(val): return False
    return True 

@njit(fastmath=True)
def normalize_l2(emb_matrix):
    rows, cols = emb_matrix.shape
    normalized = np.zeros_like(emb_matrix)
    for i in range(rows):
        norm = 0.0
        for j in range(cols):
            norm += emb_matrix[i, j] ** 2
        norm = np.sqrt(norm)
        if norm > 1e-8:
            for j in range(cols):
                normalized[i, j] = emb_matrix[i, j] / norm
        else:
            for j in range(cols):
                normalized[i, j] = emb_matrix[i, j]
    return normalized

# ==========================================
# 2. DATA PREPARATION (OFFICIAL EMBEDDINGS)
# ==========================================
def setup_and_prepare_task2():
    log(f"\n[PHASE 1] Preparing Task 2 Data (Emb ID={Config.SELECTED_EMB_ID})...")
    
    # Setup Repo
    if not os.path.exists(Config.REPO_DIR):
        os.system(f"git clone {Config.REPO_URL} {Config.REPO_DIR}")
    os.system(f"pip install -q -r {Config.REPO_DIR}/requirements.txt")
    os.system("pip install polars==0.20.31")
    
    os.makedirs(Config.DEST_DATA_DIR, exist_ok=True)

    # 1. Extract Official Embedding using Polars (Fast)
    log(f"   > Extracting Embedding column...")
    df_emb_all = pl.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "item_emb.parquet"))
    
    # Find column name (e.g. item_emb_2)
    target_col = next((c for c in df_emb_all.columns if str(Config.SELECTED_EMB_ID) in c and "emb" in c), None)
    if not target_col: raise ValueError(f"Embedding {Config.SELECTED_EMB_ID} not found.")
    
    # 2. Convert to Pandas for safe merging & cleaning
    # We only take ID and the target embedding
    pdf_emb = df_emb_all.select(["item_id", target_col]).to_pandas()
    pdf_emb = pdf_emb.rename(columns={target_col: "item_emb_d128"})
    pdf_emb["item_id"] = pdf_emb["item_id"].astype("int64")
    
    # Load Item Info (Tags)
    pdf_info = pd.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "item_info.parquet"))
    pdf_info["item_id"] = pdf_info["item_id"].astype("int64")
    # Drop existing embeddings in item_info if any
    cols_to_keep = [c for c in pdf_info.columns if "emb" not in c]
    pdf_info = pdf_info[cols_to_keep]
    
    # Merge
    pdf_final = pd.merge(pdf_info, pdf_emb, on="item_id", how="left")
    
    # 3. Clean & Normalize
    zero_vec = [0.0] * 128
    
    def clean_vec(x):
        if x is None: return zero_vec
        if isinstance(x, (np.ndarray, list)):
            if len(x) != 128: return zero_vec
            return [float(i) for i in x]
        return zero_vec

    log("   > Normalizing Vectors...")
    # Clean list structure
    cleaned_series = pdf_final["item_emb_d128"].apply(clean_vec)
    
    # Numba Normalize
    mat = np.stack(cleaned_series.values).astype(np.float32)
    mat = normalize_l2(mat)
    
    # Back to DataFrame
    pdf_final["item_emb_d128"] = list(mat)
    
    # Keep only needed columns
    pdf_final = pdf_final[["item_id", "item_tags", "item_emb_d128"]]

    # 4. Enrich Datasets
    for split in ["train", "valid", "test"]:
        log(f"   > Enriching {split}.parquet...")
        src = os.path.join(Config.SOURCE_DATA_DIR, f"{split}.parquet")
        dst = os.path.join(Config.DEST_DATA_DIR, f"{split}.parquet")
        
        pdf_data = pd.read_parquet(src)
        pdf_data["item_id"] = pdf_data["item_id"].astype("int64")
        
        # Merge
        pdf_merged = pdf_data.merge(pdf_final, on="item_id", how="left")
        
        # Fill Nulls
        null_mask_tags = pdf_merged["item_tags"].isnull()
        if null_mask_tags.any():
            pdf_merged.loc[null_mask_tags, "item_tags"] = pd.Series([[0]*5] * null_mask_tags.sum(), index=pdf_merged.index[null_mask_tags])

        null_mask_emb = pdf_merged["item_emb_d128"].isnull()
        if null_mask_emb.any():
            pdf_merged.loc[null_mask_emb, "item_emb_d128"] = pd.Series([zero_vec] * null_mask_emb.sum(), index=pdf_merged.index[null_mask_emb])

        # Save via Polars with Strict Schema
        pl_merged = pl.from_pandas(pdf_merged)
        try:
            pl_merged = pl_merged.with_columns([
                pl.col("item_emb_d128").cast(pl.Array(pl.Float32, 128)),
                pl.col("item_tags").cast(pl.List(pl.Int64))
            ])
        except:
            pl_merged = pl_merged.with_columns([
                pl.col("item_emb_d128").cast(pl.List(pl.Float32)),
                pl.col("item_tags").cast(pl.List(pl.Int64))
            ])
            
        pl_merged.write_parquet(dst)
        del pdf_data, pdf_merged, pl_merged
        gc.collect()

    # Save item_info
    pl_info = pl.from_pandas(pdf_final)
    try:
        pl_info = pl_info.with_columns([
            pl.col("item_emb_d128").cast(pl.Array(pl.Float32, 128)),
            pl.col("item_tags").cast(pl.List(pl.Int64))
        ])
    except:
        pl_info = pl_info.with_columns([
            pl.col("item_emb_d128").cast(pl.List(pl.Float32)),
            pl.col("item_tags").cast(pl.List(pl.Int64))
        ])
    pl_info.write_parquet(os.path.join(Config.DEST_DATA_DIR, "item_info.parquet"))
    
    log("✅ Task 2 Data Ready.")

# ==========================================
# 3. CONFIGURATION (OPTIMIZED)
# ==========================================
def create_config():
    log("\n[PHASE 2] Generating Config...")
    abs_root = os.path.abspath(Config.DEST_DATA_DIR)
    config_dir = os.path.abspath(os.path.join(Config.REPO_DIR, "config"))
    os.makedirs(config_dir, exist_ok=True)

    dataset_cfg = {
        Config.DATASET_ID: {
            "data_root": abs_root,
            "data_format": "parquet",
            "train_data": os.path.join(abs_root, "train.parquet"),
            "valid_data": os.path.join(abs_root, "valid.parquet"),
            "test_data": os.path.join(abs_root, "test.parquet"),
            "item_info": os.path.join(abs_root, "item_info.parquet"),
        }
    }
    
    model_cfg = {
        "base_expid": "DIN_task2_tuned",
        "Base": {
            "model_root": "./checkpoints/",
            "num_workers": 3,
            "verbose": 1,
            "debug_mode": False,
            "pickle_feature_encoder": True,
            "use_hdf5": False,
            "save_best_only": True,
            "monitor": "AUC",
            "monitor_mode": "max",
            "early_stop_patience": 4, 
        },
        "DIN_task2_tuned": {
            "model": "DIN",
            "dataset_id": Config.DATASET_ID,
            "loss": "binary_crossentropy",
            "metrics": ["AUC", "logloss"],
            "task": "binary_classification",
            "optimizer": "adam",
            "learning_rate": 5e-4, 
            "embedding_regularizer": 5e-7,
            "net_regularizer": 0,
            "batch_size": 4096,
            "embedding_dim": 64,
            "dnn_hidden_units": [2048, 1024, 512, 256],
            "dnn_activations": "ReLU",
            "attention_hidden_units": [1024, 512, 256],
            "attention_hidden_activations": "ReLU",
            "attention_output_activation": None,
            "attention_dropout": 0.2,
            "net_dropout": 0.2,
            "batch_norm": True,
            "epochs": 8,
            "shuffle": True,
            "seed": 2025,
            "feature_cols": [
                {'active': True, 'dtype': 'int', 'name': 'user_id', 'type': 'meta'},
                {'active': True, 'dtype': 'int', 'name': 'item_seq', 'type': 'meta'},
                {'active': True, 'dtype': 'int', 'name': 'likes_level', 'type': 'categorical', 'vocab_size': 11},
                {'active': True, 'dtype': 'int', 'name': 'views_level', 'type': 'categorical', 'vocab_size': 11},
                {'active': True, 'dtype': 'int', 'name': 'item_id', 'type': 'categorical', 'vocab_size': 91718},
                {'active': True, 'dtype': 'int', 'max_len': 5, 'name': 'item_tags', 'type': 'sequence', 'vocab_size': 11740},
                {'active': True, 'dtype': 'float', 'embedding_dim': 128, 'name': 'item_emb_d128', 'type': 'embedding'}
            ]
        },
        "tuner_space": {"model_root": ["./checkpoints/"]}
    }
    
    with open(os.path.join(config_dir, "dataset_config.yaml"), 'w') as f:
        yaml.dump(dataset_cfg, f)
    
    model_path = os.path.join(config_dir, "model_config.yaml")
    with open(model_path, 'w') as f:
        yaml.dump(model_cfg, f)
        
    return model_path

# ==========================================
# 4. TRAINING
# ==========================================
def run_training(config_path):
    log("\n[PHASE 3] Training Task 2...")
    
    # Cleanup
    for f in glob.glob(os.path.join(Config.DEST_DATA_DIR, "*.json")) + glob.glob(os.path.join(Config.DEST_DATA_DIR, "*.pkl")):
        os.remove(f)
        
    cwd = os.getcwd()
    os.chdir(Config.REPO_DIR)
    
    cmd = f"python run_param_tuner.py --config {config_path} --gpu 0"
    os.system(cmd)
    
    os.chdir(cwd)
    
    # Find Best Model
    ckpt = os.path.join(Config.REPO_DIR, "checkpoints", Config.DATASET_ID)
    models = glob.glob(os.path.join(ckpt, "*.model"))
    if not models:
        log("🔴 Training Failed or Interrupted (No model file).")
        return False
        
    best = max(models, key=os.path.getmtime)
    shutil.copy(best, Config.BEST_MODEL_DEST)
    log(f"✅ Best Model Saved: {Config.BEST_MODEL_DEST}")
    return True

# ==========================================
# 5. GOLDEN INFERENCE
# ==========================================
def load_din(path):
    spec = importlib.util.spec_from_file_location("DIN_C", path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules["DIN_C"] = mod
    spec.loader.exec_module(mod)
    return getattr(mod, "DIN")

def pad_seq(s_list, max_len=50):
    res = []
    for s in s_list:
        if not isinstance(s, (list, np.ndarray)): s = []
        s = list(s)
        if len(s) > max_len: s = s[-max_len:]
        else: s = s + [0]*(max_len-len(s))
        res.append(s)
    return torch.tensor(res, dtype=torch.long).to("cuda")

def run_inference():
    log("\n[PHASE 4] Inference Task 2 (Flattened Logic)...")
    from fuxictr.features import FeatureMap
    
    params = {
        "model_id": "DIN_task2_tuned", "dataset_id": Config.DATASET_ID,
        "data_root": Config.DEST_DATA_DIR, "model_root": Config.WORKING_DIR,
        "feature_cols": [], "dnn_hidden_units": [2048, 1024, 512, 256],
        "attention_hidden_units": [1024, 512, 256], "dnn_activations": "ReLU",
        "attention_hidden_activations": "ReLU", "embedding_dim": 64,
        "batch_norm": True, "din_use_softmax": False, "gpu": 0,
        "task": "binary_classification", "loss": "binary_crossentropy",
        "metrics": ["AUC", "logloss"], "optimizer": "adam", "learning_rate": 5e-4,
        "verbose": 0, "attention_dropout": 0.2, "net_dropout": 0.2
    }

    fmap = FeatureMap(Config.DATASET_ID, Config.DEST_DATA_DIR)
    
    # Handle FuxiCTR internal structure change
    inner = os.path.join(Config.DEST_DATA_DIR, "MicroLens_1M_x1", "feature_map.json")
    outer = os.path.join(Config.DEST_DATA_DIR, "feature_map.json")
    if os.path.exists(inner): fmap.load(inner, params)
    else: fmap.load(outer, params)
    
    DIN = load_din(os.path.join(Config.REPO_DIR, "src", "DIN.py"))
    model = DIN(fmap, **params)
    model.load_state_dict(torch.load(Config.BEST_MODEL_DEST, map_location="cuda:0"))
    model.to("cuda")
    model.eval()

    # Load Data (Pre-merged)
    df_test = pl.read_parquet(os.path.join(Config.DEST_DATA_DIR, "test.parquet"))
    
    # History
    df_seq = pl.read_parquet(os.path.join(Config.SOURCE_DATA_DIR, "item_seq.parquet"))
    if "item_seq" not in df_seq.columns: 
        c = [x for x in df_seq.columns if x != 'user_id'][0]
        df_seq = df_seq.rename({c: "item_seq"})
    
    df_seq = df_seq.select(["user_id", "item_seq"]).with_columns(pl.col("user_id").cast(pl.Int64))
    df_seq = df_seq.unique(subset=["user_id"], keep="first")

    # Join
    df = df_test.join(df_seq, on="user_id", how="left")
    log(f"    Rows to Predict: {len(df)}")
    
    all_preds = []
    with torch.no_grad():
        for start in tqdm(range(0, len(df), Config.BATCH_SIZE)):
            batch = df.slice(start, Config.BATCH_SIZE)
            bs = len(batch)
            
            # 1. Batch Input
            bi = {}
            if 'user_id' in batch.columns:
                bi['user_id'] = torch.tensor(batch['user_id'].to_numpy(), dtype=torch.long).to("cuda")
            for c in ['likes_level', 'views_level']:
                if c in batch.columns:
                    bi[c] = torch.tensor(batch[c].fill_null(0).to_numpy(), dtype=torch.long).to("cuda")

            # 2. Item Input (Flattened)
            hist = pad_seq(batch['item_seq'].to_list())
            tgt_id = torch.tensor(batch['item_id'].to_numpy(), dtype=torch.long).to("cuda").unsqueeze(1)
            
            # IDs
            ii = {'item_id': torch.cat([hist, tgt_id], dim=1).view(-1)} 
            
            # Tags
            if 'item_tags' in batch.columns:
                tags = batch['item_tags'].to_list()
                clean_t = []
                for t in tags:
                    if t is None: t = []
                    t = list(t)
                    if len(t)>5: t=t[:5]
                    else: t=t+[0]*(5-len(t))
                    clean_t.append(t)
                tgt_tag = torch.tensor(clean_t, dtype=torch.long).to("cuda").unsqueeze(1)
                dum_tag = torch.zeros((bs, 50, 5), dtype=torch.long).to("cuda")
                ii['item_tags'] = torch.cat([dum_tag, tgt_tag], dim=1).view(-1, 5)

            # Embeddings
            if 'item_emb_d128' in batch.columns:
                embs = batch['item_emb_d128'].to_list()
                clean_e = [e if e is not None and len(e)==128 else [0.0]*128 for e in embs]
                tgt_e = torch.tensor(np.array(clean_e, dtype=np.float32)).to("cuda").unsqueeze(1)
                dum_e = torch.zeros((bs, 50, 128), dtype=torch.float32).to("cuda")
                ii['item_emb_d128'] = torch.cat([dum_e, tgt_e], dim=1).view(-1, 128)

            # Mask
            mask = (hist > 0).float().to("cuda")
            
            out = model.forward((bi, ii, mask))
            p = torch.sigmoid(out['y_pred'] if isinstance(out, dict) else out)
            all_preds.extend(p.cpu().numpy().flatten())

    sub = pd.DataFrame({"ID": range(len(all_preds)), "Task2": all_preds})
    sub.to_csv(Config.FINAL_SUBMISSION, index=False)
    log(f"✅ Prediction Saved: {Config.FINAL_SUBMISSION}")

if __name__ == "__main__":
    setup_and_prepare_task2()
    cfg = create_config()
    if run_training(cfg):
        run_inference()